In [6]:
pip install kagglehub

StatementMeta(, 1330ef20-82c9-43a5-9531-2a3eaf90fc84, 8, Finished, Available, Finished, False)

Note: you may need to restart the kernel to use updated packages.


In [7]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("martinellis/nhl-game-data")

print("Path to dataset files:", path)

StatementMeta(, 1330ef20-82c9-43a5-9531-2a3eaf90fc84, 9, Finished, Available, Finished, False)

ImportError: cannot import name 'get_web_endpoint' from 'kagglesdk.kaggle_env' (/home/trusted-service-user/cluster-env/trident_env/lib/python3.11/site-packages/kagglesdk/kaggle_env.py)

In [ ]:
import os

path = "/home/trusted-service-user/.cache/kagglehub/datasets/martinellis/nhl-game-data/versions/2"

files = os.listdir(path)

for file in files:
    print(file)

StatementMeta(, 1330ef20-82c9-43a5-9531-2a3eaf90fc84, -1, Cancelled, , Cancelled, True)

In [ ]:
pip install kagglehub

StatementMeta(, 1330ef20-82c9-43a5-9531-2a3eaf90fc84, -1, Cancelled, , Cancelled, True)

In [ ]:
import kagglehub
import os

path = kagglehub.dataset_download("martinellis/nhl-game-data")

print("Dataset path:", path)
print(os.listdir(path))

StatementMeta(, 1330ef20-82c9-43a5-9531-2a3eaf90fc84, -1, Cancelled, , Cancelled, True)

In [ ]:
import os
import pandas as pd

csv_files = [file for file in os.listdir(path) if file.endswith(".csv")]

for file in csv_files:
    file_path = f"{path}/{file}"
    table_name = file.replace(".csv", "").lower()
    
    print(f"Loading {file} into table: {table_name}")
    
    pdf = pd.read_csv(file_path)
    sdf = spark.createDataFrame(pdf)
    
    sdf.write.mode("overwrite").saveAsTable(table_name)
    
    print(f"Saved table: {table_name}")

print("All NHL CSV files loaded successfully.")

StatementMeta(, 1330ef20-82c9-43a5-9531-2a3eaf90fc84, -1, Cancelled, , Cancelled, True)

In [ ]:
spark.sql("SHOW TABLES").show(truncate=False)

StatementMeta(, 1330ef20-82c9-43a5-9531-2a3eaf90fc84, -1, Cancelled, , Cancelled, True)

In [ ]:
for table in spark.sql("SHOW TABLES").select("tableName").collect():
    table_name = table["tableName"]
    count = spark.table(table_name).count()
    print(table_name, count)

StatementMeta(, 1330ef20-82c9-43a5-9531-2a3eaf90fc84, -1, Cancelled, , Cancelled, True)

In [ ]:
tables = [
    "game",
    "team_info",
    "player_info",
    "game_teams_stats",
    "game_skater_stats",
    "game_goalie_stats",
    "game_plays",
    "game_goals",
    "game_penalties"
]

for table in tables:
    print(f"\n===== {table} =====")
    spark.table(table).printSchema()

StatementMeta(, 1330ef20-82c9-43a5-9531-2a3eaf90fc84, -1, Cancelled, , Cancelled, True)

In [ ]:
display(spark.table("game").limit(5))
display(spark.table("team_info").limit(5))
display(spark.table("player_info").limit(5))
display(spark.table("game_teams_stats").limit(5))
display(spark.table("game_skater_stats").limit(5))
display(spark.table("game_plays").limit(5))

StatementMeta(, 1330ef20-82c9-43a5-9531-2a3eaf90fc84, -1, Cancelled, , Cancelled, True)

In [ ]:
from pyspark.sql.functions import col, to_date

silver_game = (
    spark.table("game")
    .dropDuplicates(["game_id"])
    .withColumn("date_time_gmt", col("date_time_gmt").cast("timestamp"))
    .withColumn("game_date", to_date(col("date_time_gmt")))
)

silver_game.write.mode("overwrite").saveAsTable("silver_game")

StatementMeta(, 1330ef20-82c9-43a5-9531-2a3eaf90fc84, -1, Cancelled, , Cancelled, True)

In [ ]:
silver_team = (
    spark.table("team_info")
    .dropDuplicates(["team_id"])
)

silver_team.write.mode("overwrite").saveAsTable("silver_team")

StatementMeta(, 1330ef20-82c9-43a5-9531-2a3eaf90fc84, -1, Cancelled, , Cancelled, True)

In [ ]:
silver_player = (
    spark.table("player_info")
    .dropDuplicates(["player_id"])
)

silver_player.write.mode("overwrite").saveAsTable("silver_player")

StatementMeta(, 1330ef20-82c9-43a5-9531-2a3eaf90fc84, -1, Cancelled, , Cancelled, True)

In [ ]:
silver_team_game_stats = (
    spark.table("game_teams_stats")
    .dropDuplicates(["game_id", "team_id"])
)

silver_team_game_stats.write.mode("overwrite").saveAsTable("silver_team_game_stats")

StatementMeta(, 1330ef20-82c9-43a5-9531-2a3eaf90fc84, -1, Cancelled, , Cancelled, True)

In [ ]:
spark.sql("SHOW TABLES").show(truncate=False)

StatementMeta(, 1330ef20-82c9-43a5-9531-2a3eaf90fc84, -1, Cancelled, , Cancelled, True)

In [ ]:
# Check duplicate game IDs
spark.sql("""
SELECT game_id, COUNT(*) AS count
FROM silver_game
GROUP BY game_id
HAVING COUNT(*) > 1
""").show()

StatementMeta(, 1330ef20-82c9-43a5-9531-2a3eaf90fc84, -1, Cancelled, , Cancelled, True)

In [ ]:
# Check duplicate team IDs
spark.sql("""
SELECT team_id, COUNT(*) AS count
FROM silver_team
GROUP BY team_id
HAVING COUNT(*) > 1
""").show()

StatementMeta(, 1330ef20-82c9-43a5-9531-2a3eaf90fc84, -1, Cancelled, , Cancelled, True)

In [ ]:
# Check missing game references in team stats
spark.sql("""
SELECT COUNT(*) AS missing_game_ids
FROM silver_team_game_stats t
LEFT JOIN silver_game g
ON t.game_id = g.game_id
WHERE g.game_id IS NULL
""").show()

StatementMeta(, 1330ef20-82c9-43a5-9531-2a3eaf90fc84, -1, Cancelled, , Cancelled, True)

In [ ]:
spark.sql("SHOW TABLES").show(truncate=False)

StatementMeta(, 1330ef20-82c9-43a5-9531-2a3eaf90fc84, -1, Cancelled, , Cancelled, True)

In [ ]:
spark.table("silver_team_game_stats").printSchema()
display(spark.table("silver_team_game_stats").limit(5))

StatementMeta(, 1330ef20-82c9-43a5-9531-2a3eaf90fc84, -1, Cancelled, , Cancelled, True)

In [ ]:
from pyspark.sql.functions import col, sum, count, round

gold_team_performance = (
    spark.table("silver_team_game_stats")
    .groupBy("team_id")
    .agg(
        count("game_id").alias("games_played"),
        sum(col("won").cast("int")).alias("wins"),
        sum(col("goals")).alias("goals_for"),
        sum(col("shots")).alias("shots_for")
    )
    .withColumn("losses", col("games_played") - col("wins"))
    .withColumn("win_percentage", round(col("wins") / col("games_played") * 100, 2))
    .withColumn("avg_goals_per_game", round(col("goals_for") / col("games_played"), 2))
    .withColumn("avg_shots_per_game", round(col("shots_for") / col("games_played"), 2))
)

gold_team_performance.write.mode("overwrite").saveAsTable("gold_team_performance")

StatementMeta(, 1330ef20-82c9-43a5-9531-2a3eaf90fc84, -1, Cancelled, , Cancelled, True)

In [ ]:
gold_team_performance_named = (
    spark.table("gold_team_performance").alias("g")
    .join(
        spark.table("silver_team").alias("t"),
        col("g.team_id") == col("t.team_id"),
        "left"
    )
    .select(
        col("g.team_id"),
        col("t.shortName").alias("team_name"),
        col("t.teamName").alias("team_full_name"),
        col("g.games_played"),
        col("g.wins"),
        col("g.losses"),
        col("g.goals_for"),
        col("g.shots_for"),
        col("g.win_percentage"),
        col("g.avg_goals_per_game"),
        col("g.avg_shots_per_game")
    )
)

gold_team_performance_named.write.mode("overwrite").saveAsTable("gold_team_performance_named")

StatementMeta(, 1330ef20-82c9-43a5-9531-2a3eaf90fc84, -1, Cancelled, , Cancelled, True)

In [ ]:
display(
    spark.table("gold_team_performance_named")
    .orderBy(col("win_percentage").desc())
)

StatementMeta(, 1330ef20-82c9-43a5-9531-2a3eaf90fc84, -1, Cancelled, , Cancelled, True)

In [ ]:
spark.table("silver_team_game_stats").printSchema()
display(spark.table("silver_team_game_stats").limit(5))

StatementMeta(, 1330ef20-82c9-43a5-9531-2a3eaf90fc84, -1, Cancelled, , Cancelled, True)

In [ ]:
from pyspark.sql.functions import col, sum, count, round

gold_team_performance = (
    spark.table("silver_team_game_stats")
    .groupBy("team_id")
    .agg(
        count("game_id").alias("games_played"),
        sum(col("won").cast("int")).alias("wins"),
        sum("goals").alias("goals_for"),
        sum("shots").alias("shots_for"),
        sum("hits").alias("total_hits"),
        sum("pim").alias("penalty_minutes"),
        sum("powerPlayGoals").alias("power_play_goals"),
        sum("powerPlayOpportunities").alias("power_play_opportunities"),
        sum("giveaways").alias("giveaways"),
        sum("takeaways").alias("takeaways")
    )
    .withColumn("losses", col("games_played") - col("wins"))
    .withColumn("win_percentage", round((col("wins") / col("games_played")) * 100, 2))
    .withColumn("avg_goals_per_game", round(col("goals_for") / col("games_played"), 2))
    .withColumn("avg_shots_per_game", round(col("shots_for") / col("games_played"), 2))
    .withColumn("power_play_success_rate", round((col("power_play_goals") / col("power_play_opportunities")) * 100, 2))
)

gold_team_performance.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold_team_performance")

StatementMeta(, 1330ef20-82c9-43a5-9531-2a3eaf90fc84, -1, Cancelled, , Cancelled, True)

In [ ]:
display(
    spark.table("gold_team_performance")
    .orderBy(col("win_percentage").desc())
)

In [ ]:
spark.table("silver_team").printSchema()
display(spark.table("silver_team").limit(5))

StatementMeta(, 1330ef20-82c9-43a5-9531-2a3eaf90fc84, -1, Cancelled, , Cancelled, True)

In [27]:
from pyspark.sql.functions import col

gold_team_performance_named = (
    spark.table("gold_team_performance").alias("g")
    .join(
        spark.table("silver_team").alias("t"),
        col("g.team_id") == col("t.team_id"),
        "left"
    )
    .select(
        col("g.team_id"),
        col("t.shortName").alias("team_location"),
        col("t.teamName").alias("team_name"),
        col("t.abbreviation"),
        col("g.games_played"),
        col("g.wins"),
        col("g.losses"),
        col("g.goals_for"),
        col("g.shots_for"),
        col("g.total_hits"),
        col("g.penalty_minutes"),
        col("g.power_play_goals"),
        col("g.power_play_opportunities"),
        col("g.giveaways"),
        col("g.takeaways"),
        col("g.win_percentage"),
        col("g.avg_goals_per_game"),
        col("g.avg_shots_per_game"),
        col("g.power_play_success_rate")
    )
)

gold_team_performance_named.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold_team_performance_named")

StatementMeta(, f405e6b2-ee0c-45f6-a051-70f7492e7fc4, 29, Finished, Available, Finished, False)

In [28]:
display(
    spark.table("gold_team_performance_named")
    .orderBy(col("win_percentage").desc())
)

StatementMeta(, f405e6b2-ee0c-45f6-a051-70f7492e7fc4, 30, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 68183fc2-ad76-4c12-ad14-b8599915794a)

In [30]:
from pyspark.sql.functions import col, sum, count, round

gold_player_performance = (
    spark.table("silver_player").alias("p")
    .join(
        spark.table("game_skater_stats").alias("s"),
        col("p.player_id") == col("s.player_id"),
        "inner"
    )
    .groupBy(
        col("p.player_id"),
        col("p.firstName"),
        col("p.lastName"),
        col("p.nationality"),
        col("p.primaryPosition")
    )
    .agg(
        count("s.game_id").alias("games_played"),
        sum("s.goals").alias("goals"),
        sum("s.assists").alias("assists"),
        sum("s.shots").alias("shots"),
        sum("s.hits").alias("hits"),
        sum("s.penaltyMinutes").alias("penalty_minutes"),
        sum("s.powerPlayGoals").alias("power_play_goals"),
        sum("s.powerPlayAssists").alias("power_play_assists"),
        sum("s.takeaways").alias("takeaways"),
        sum("s.giveaways").alias("giveaways"),
        sum("s.blocked").alias("blocked_shots")
    )
    .withColumn("points", col("goals") + col("assists"))
    .withColumn("avg_points_per_game", round(col("points") / col("games_played"), 2))
    .withColumn("shot_conversion_rate", round((col("goals") / col("shots")) * 100, 2))
)

gold_player_performance.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold_player_performance")

StatementMeta(, f405e6b2-ee0c-45f6-a051-70f7492e7fc4, 32, Finished, Available, Finished, False)

In [31]:
display(
    spark.table("gold_player_performance")
    .orderBy(col("points").desc())
)

StatementMeta(, f405e6b2-ee0c-45f6-a051-70f7492e7fc4, 33, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, fd0a6983-6f36-4307-b612-bfa70fd4a342)

In [32]:
spark.table("silver_game").printSchema()
display(spark.table("silver_game").limit(5))

StatementMeta(, f405e6b2-ee0c-45f6-a051-70f7492e7fc4, 34, Finished, Available, Finished, False)

root
 |-- game_id: long (nullable = true)
 |-- season: long (nullable = true)
 |-- type: string (nullable = true)
 |-- date_time_gmt: timestamp (nullable = true)
 |-- away_team_id: long (nullable = true)
 |-- home_team_id: long (nullable = true)
 |-- away_goals: long (nullable = true)
 |-- home_goals: long (nullable = true)
 |-- outcome: string (nullable = true)
 |-- home_rink_side_start: string (nullable = true)
 |-- venue: string (nullable = true)
 |-- venue_link: string (nullable = true)
 |-- venue_time_zone_id: string (nullable = true)
 |-- venue_time_zone_offset: long (nullable = true)
 |-- venue_time_zone_tz: string (nullable = true)
 |-- game_date: date (nullable = true)



SynapseWidget(Synapse.DataFrame, c1590feb-9dfa-4229-8395-78159ecd60ab)

In [33]:
from pyspark.sql.functions import col

gold_game_summary = (
    spark.table("silver_game").alias("g")
    .join(
        spark.table("silver_team").alias("home"),
        col("g.home_team_id") == col("home.team_id"),
        "left"
    )
    .join(
        spark.table("silver_team").alias("away"),
        col("g.away_team_id") == col("away.team_id"),
        "left"
    )
    .select(
        col("g.game_id"),
        col("g.season"),
        col("g.type"),
        col("g.game_date"),
        col("g.date_time_gmt"),
        col("home.teamName").alias("home_team"),
        col("away.teamName").alias("away_team"),
        col("g.home_goals"),
        col("g.away_goals"),
        col("g.outcome"),
        col("g.venue")
    )
)

gold_game_summary.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold_game_summary")

StatementMeta(, f405e6b2-ee0c-45f6-a051-70f7492e7fc4, 35, Finished, Available, Finished, False)

In [34]:
display(
    spark.table("gold_game_summary")
)

StatementMeta(, f405e6b2-ee0c-45f6-a051-70f7492e7fc4, 36, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 71196398-d928-47b2-b608-9291edaa912e)

# Data Validation & Testing

In [35]:
spark.sql("""
SELECT game_id, COUNT(*) AS duplicate_count
FROM silver_game
GROUP BY game_id
HAVING COUNT(*) > 1
""").show()

StatementMeta(, f405e6b2-ee0c-45f6-a051-70f7492e7fc4, 37, Finished, Available, Finished, False)

+-------+---------------+
|game_id|duplicate_count|
+-------+---------------+
+-------+---------------+



In [36]:
spark.sql("""
SELECT player_id, COUNT(*) AS duplicate_count
FROM silver_player
GROUP BY player_id
HAVING COUNT(*) > 1
""").show()

StatementMeta(, f405e6b2-ee0c-45f6-a051-70f7492e7fc4, 38, Finished, Available, Finished, False)

+---------+---------------+
|player_id|duplicate_count|
+---------+---------------+
+---------+---------------+



In [37]:
spark.sql("""
SELECT COUNT(*) AS missing_team_records
FROM silver_team_game_stats t
LEFT JOIN silver_team s
ON t.team_id = s.team_id
WHERE s.team_id IS NULL
""").show()

StatementMeta(, f405e6b2-ee0c-45f6-a051-70f7492e7fc4, 39, Finished, Available, Finished, False)

+--------------------+
|missing_team_records|
+--------------------+
|                  10|
+--------------------+



In [39]:
spark.sql("""
SELECT 
    t.team_id,
    COUNT(*) AS missing_count
FROM silver_team_game_stats t
LEFT JOIN silver_team s
ON t.team_id = s.team_id
WHERE s.team_id IS NULL
GROUP BY t.team_id
ORDER BY missing_count DESC
""").show()

StatementMeta(, f405e6b2-ee0c-45f6-a051-70f7492e7fc4, 41, Finished, Available, Finished, False)

+-------+-------------+
|team_id|missing_count|
+-------+-------------+
|     87|            3|
|     88|            3|
|     89|            2|
|     90|            2|
+-------+-------------+



In [40]:
spark.sql("""
SELECT *
FROM silver_team_game_stats
WHERE team_id NOT IN (
    SELECT team_id FROM silver_team
)
""").show(truncate=False)

StatementMeta(, f405e6b2-ee0c-45f6-a051-70f7492e7fc4, 42, Finished, Available, Finished, False)

+----------+-------+----+-----+----------+-------------+-----+-----+----+---+----------------------+--------------+--------------------+---------+---------+-------+-------------+
|game_id   |team_id|HoA |won  |settled_in|head_coach   |goals|shots|hits|pim|powerPlayOpportunities|powerPlayGoals|faceOffWinPercentage|giveaways|takeaways|blocked|startRinkSide|
+----------+-------+----+-----+----------+-------------+-----+-----+----+---+----------------------+--------------+--------------------+---------+---------+-------+-------------+
|2018040642|88     |away|true |tbc       |Todd Reirden |7.0  |26.0 |0.0 |0.0|0.0                   |0.0           |56.2                |2.0      |6.0      |1.0    |NULL         |
|2018040643|89     |home|false|tbc       |Paul Maurice |5.0  |23.0 |0.0 |0.0|0.0                   |0.0           |55.0                |4.0      |4.0      |2.0    |NULL         |
|2019040651|87     |home|true |tbc       |Bruce Cassidy|9.0  |22.0 |1.0 |0.0|0.0                   |0.0  

A referential integrity check identified 10 team-stat records where the team_id did not match the team reference table. These records were flagged for review to prevent inaccurate team-level reporting.

In [41]:
silver_team_game_stats_valid = (
    spark.table("silver_team_game_stats").alias("t")
    .join(
        spark.table("silver_team").alias("s"),
        col("t.team_id") == col("s.team_id"),
        "inner"
    )
    .select("t.*")
)

silver_team_game_stats_valid.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver_team_game_stats_valid")

StatementMeta(, f405e6b2-ee0c-45f6-a051-70f7492e7fc4, 43, Finished, Available, Finished, False)

In [42]:
spark.sql("""
SELECT
    SUM(CASE WHEN game_id IS NULL THEN 1 ELSE 0 END) AS null_game_ids,
    SUM(CASE WHEN home_team_id IS NULL THEN 1 ELSE 0 END) AS null_home_teams,
    SUM(CASE WHEN away_team_id IS NULL THEN 1 ELSE 0 END) AS null_away_teams
FROM silver_game
""").show()

StatementMeta(, f405e6b2-ee0c-45f6-a051-70f7492e7fc4, 44, Finished, Available, Finished, False)

+-------------+---------------+---------------+
|null_game_ids|null_home_teams|null_away_teams|
+-------------+---------------+---------------+
|            0|              0|              0|
+-------------+---------------+---------------+



In [1]:
from pyspark.sql.functions import col, sum, count, round

gold_team_performance = (
    spark.table("silver_team_game_stats_valid")
    .groupBy("team_id")
    .agg(
        count("game_id").alias("games_played"),
        sum(col("won").cast("int")).alias("wins"),
        sum("goals").alias("goals_for"),
        sum("shots").alias("shots_for"),
        sum("hits").alias("total_hits"),
        sum("pim").alias("penalty_minutes"),
        sum("powerPlayGoals").alias("power_play_goals"),
        sum("powerPlayOpportunities").alias("power_play_opportunities"),
        sum("giveaways").alias("giveaways"),
        sum("takeaways").alias("takeaways")
    )
    .withColumn("losses", col("games_played") - col("wins"))
    .withColumn("win_percentage", round((col("wins") / col("games_played")) * 100, 2))
    .withColumn("avg_goals_per_game", round(col("goals_for") / col("games_played"), 2))
    .withColumn("avg_shots_per_game", round(col("shots_for") / col("games_played"), 2))
    .withColumn("power_play_success_rate", round((col("power_play_goals") / col("power_play_opportunities")) * 100, 2))
)

gold_team_performance.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold_team_performance")

StatementMeta(, b247d658-9991-45fc-8f25-0cde51ed504c, 3, Finished, Available, Finished, False)

In [2]:
gold_team_performance_named = (
    spark.table("gold_team_performance").alias("g")
    .join(
        spark.table("silver_team").alias("t"),
        col("g.team_id") == col("t.team_id"),
        "left"
    )
    .select(
        col("g.team_id"),
        col("t.shortName").alias("team_location"),
        col("t.teamName").alias("team_name"),
        col("t.abbreviation"),
        col("g.games_played"),
        col("g.wins"),
        col("g.losses"),
        col("g.goals_for"),
        col("g.shots_for"),
        col("g.total_hits"),
        col("g.penalty_minutes"),
        col("g.power_play_goals"),
        col("g.power_play_opportunities"),
        col("g.giveaways"),
        col("g.takeaways"),
        col("g.win_percentage"),
        col("g.avg_goals_per_game"),
        col("g.avg_shots_per_game"),
        col("g.power_play_success_rate")
    )
)

gold_team_performance_named.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold_team_performance_named")

StatementMeta(, b247d658-9991-45fc-8f25-0cde51ed504c, 4, Finished, Available, Finished, False)

In [3]:
spark.sql("SHOW TABLES LIKE 'gold*'").show(truncate=False)

StatementMeta(, b247d658-9991-45fc-8f25-0cde51ed504c, 5, Finished, Available, Finished, False)

+-------------+---------------------------+-----------+
|namespace    |tableName                  |isTemporary|
+-------------+---------------------------+-----------+
|nhl_lakehouse|gold_team_performance      |false      |
|nhl_lakehouse|gold_team_performance_named|false      |
|nhl_lakehouse|gold_player_performance    |false      |
|nhl_lakehouse|gold_game_summary          |false      |
+-------------+---------------------------+-----------+

